# L3: Multi-agent Customer Support Automation


- Role Playing
- Focus
- Tools
- Cooperation
- Guardrails
- Memory

In [18]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, API and LLM

In [19]:
from crewai import Agent, Task, Crew

In [20]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Role Playing, Focus and Cooperation

In [21]:
support_agent = Agent(
    role="Senior Support Representative",
    goal="Be the most friendly and helpful "
         "support representative in your team",
    backstory=(
        "You work at EIGSI and "
        "are now working on providing "
        "support to {customer}, a super important customer "
        "for your company. "
        "You need to make sure that you provide the best support! "
        "Make sure to provide full complete answers, "
        "and make no assumptions."
    ),
    allow_delegation=False,
    verbose=True
)

- By not setting `allow_delegation=False`, `allow_delegation` takes its default value of being `True`.
- This means the agent _can_ delegate its work to another agent which is better suited to do a particular task. 

In [22]:
support_quality_assurance_agent = Agent(
    role="Support Quality Assurance Specialist",
    goal="Get recognition for providing the "
         "best support quality assurance in your team",
    backstory=(
        "You work at EIGSI and "
        "are now working with your team "
        "on a request from {customer} ensuring that "
        "the support representative is "
        "providing the best support possible.\n"
        "You need to make sure that the support representative "
        "is providing full "
        "complete answers, and make no assumptions."
    ),
    verbose=True
)

* **Role Playing**: Both agents have been given a role, goal and backstory.
* **Focus**: Both agents have been prompted to get into the character of the roles they are playing.
* **Cooperation**: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.

## Tools, Guardrails and Memory

### Tools

- Import CrewAI tools

In [23]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

### Possible Custom Tools
- Load customer data
- Tap into previous conversations
- Load data from a CRM
- Checking existing bug reports
- Checking existing feature requests
- Checking ongoing tickets
- ... and more

- Some ways of using CrewAI tools.

```Python
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
```

- Instantiate a document scraper tool.
- The tool will scrape a page (only 1 URL) of the CrewAI documentation.

In [24]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://www.eigsi.fr/"
)

##### Different Ways to Give Agents Tools

- Agent Level: The Agent can use the Tool(s) on any Task it performs.
- Task Level: The Agent will only use the Tool(s) when performing that specific Task.

**Note**: Task Tools override the Agent Tools.

### Creating Tasks
- You are passing the Tool on the Task Level.

In [25]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
        "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
        "Make sure to use everything you know "
        "to provide the best support possible. "
        "You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
        "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
        "leaving no questions unanswered, and maintain a helpful and friendly "
        "tone throughout."
    ),
    tools=[docs_scrape_tool],
    agent=support_agent,
)

- `quality_assurance_review` is not using any Tool(s)
- Here the QA Agent will only review the work of the Support Agent

In [26]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
        "high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
        "thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        "find the information, "
        "ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
        "relevant feedback and improvements.\n"
        "Don't be too formal, we are a chill and cool company "
        "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)

### Creating the Crew

#### Memory
- Setting `memory=True` when putting the crew together enables Memory.

In [27]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=2,
  memory=True
)

2026-08-24 08:42:20,098 - 140403586841472 - __init__.py-__init__:518 - WARNING: Overriding of current TracerProvider is not allowed


### Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

#### Guardrails
- By running the execution below, you can see that the agents and the responses are within the scope of what we expect from them.

In [28]:
inputs = {
    "customer": "EIGSI",
    "person": "Romaric TEKEM",
    "inquiry": "I would like to get information about enrolling "
               "in the Data and Artificial Intelligence program. "
               "Could you please provide me with the admission requirements, "
               "application process, required documents, tuition fees, "
               "and important registration deadlines?"
}

result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Senior Support Representative
 [INFO]: == Starting Task: EIGSI just reached out with a super important ask:
I would like to get information about enrolling in the Data and Artificial Intelligence program. Could you please provide me with the admission requirements, application process, required documents, tuition fees, and important registration deadlines?

Romaric TEKEM from EIGSI is the one that reached out. Make sure to use everything you know to provide the best support possible. You must strive to provide a complete and accurate response to the customer's inquiry.


> Entering new CrewAgentExecutor chain...
I need to gather all the necessary information about enrolling in the Data and Artificial Intelligence program at EIGSI to provide the best support to Romaric TEKEM.

Action: Read website content
Action Input: {} 

Ecole d'ingénieurs généralistes EIGSI | Ecole ingénieur Formation | CPGE Post-Bac
Ecole Ingénieurs
Campus de Casablanca
Concours Avenir
F

Final Answer: 
Dear Romaric TEKEM,

Thank you for reaching out regarding enrolling in the Data and Artificial Intelligence program at EIGSI. Here is the information you requested:

Admission Requirements:
- To be eligible for the Data and Artificial Intelligence program at EIGSI, applicants must meet the following requirements:
  - Hold a relevant bachelor's degree in a related field.
  - Demonstrate proficiency in programming languages such as Python, R, or Java.
  - Submit a statement of purpose outlining their interest in the program.

Application Process:
- The application process for the Data and Artificial Intelligence program at EIGSI involves the following steps:
  - Submit an online application through the EIGSI website.
  - Provide all required documentation, including transcripts and letters of recommendation.
  - Attend an interview with the admissions committee.

Required Documents:
- The required documents for the application to the Data and Artificial Intelligence progra

- Display the final result as Markdown.

In [29]:
from IPython.display import Markdown
Markdown(result)

Dear Romaric TEKEM,

Thank you for your interest in the Data and Artificial Intelligence program at EIGSI. I'm here to provide you with the detailed information you requested.

Admission Requirements:
To be eligible for the program, applicants must meet the following criteria:
- Hold a relevant bachelor's degree in a related field.
- Demonstrate proficiency in programming languages such as Python, R, Java, and SQL.
- Submit a statement of purpose outlining their interest in the program.

Application Process:
The application process involves the following steps:
- Submit an online application through the EIGSI website.
- Provide transcripts, letters of recommendation, a statement of purpose, and a CV/resume.
- Attend an interview with the admissions committee.

Tuition Fees:
Tuition fees vary based on different factors. For the most up-to-date information, please contact the admissions office directly.

Registration Deadlines:
Please regularly check the EIGSI website or contact the admissions office for specific deadlines for the upcoming academic year.

If you have any further questions or need assistance, feel free to reach out to the admissions office at EIGSI. We are here to help you throughout the application process.

Best regards,

[Your Name]
Senior Support Representative
EIGSI